# Ep. 02 — Tokens, embeddings e um pipeline determinístico

LLMs não “veem” texto como caracteres ou palavras. Eles veem **tokens**. Embeddings projetam texto em vetores numéricos; isso permite comparar significado. No fim, montamos um pipeline em classes que interpreta um **ACR** (relato/caso clínico) e devolve JSON estruturado.


## 1. Tokenização

Limitação: contar **palavras** ≠ contar **tokens**. Espaços e pontuação costumam colar no pedaço anterior. Abaixo, um texto de exemplo passa pelo tokenizer BPE `cl100k_base` (família GPT-4).


In [ ]:
import tiktoken

text = "A inteligência artificial processa texto em tokens, não em palavras."
enc = tiktoken.get_encoding("cl100k_base")
token_ids = enc.encode(text)
tokens = [enc.decode([tid]) for tid in token_ids]

print("Texto:", text)
print("Tokens:", tokens)
print("Quantidade de tokens:", len(tokens))
print("Quantidade de palavras (split simples):", len(text.split()))


## 2. Embeddings (Hugging Face / sentence-transformers)

Um embedding é um vetor denso que representa o significado de um texto. Usamos o modelo leve `sentence-transformers/all-MiniLM-L6-v2` (primeira execução baixa o modelo).


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

word = "alegria"
embedding = model.encode(word)

print("Palavra:", word)
print("Dimensão do embedding:", len(embedding))
print("Primeiros 8 valores:", embedding[:8])


## 3. Comparação: significados opostos

Dois textos com sentidos opostos ficam em regiões diferentes do espaço vetorial. Medimos **similaridade de cosseno** (1 ≈ mesmo sentido; valores baixos ≈ distantes).


In [ ]:
import numpy as np

w1, w2 = "alegria", "tristeza"
e1, e2 = model.encode([w1, w2])

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

sim = cosine_similarity(e1, e2)
dist = euclidean_distance(e1, e2)

print(f"{w1!r} vs {w2!r}")
print("Similaridade de cosseno:", round(sim, 4))
print("Distância euclidiana:", round(dist, 4))


## 4. Visualização 3D (PCA)

Embeddings têm centenas de dimensões. Para **visualizar**, reduzimos a 3D com PCA (só para o gráfico — a similaridade real usa o vetor completo).


In [ ]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go

labels = ["alegria", "tristeza", "felicidade", "raiva", "calma"]
vectors = model.encode(labels)
coords = PCA(n_components=3).fit_transform(vectors)

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=coords[:, 0],
            y=coords[:, 1],
            z=coords[:, 2],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(size=6),
        )
    ]
)
fig.update_layout(
    title="Embeddings em 3D (PCA)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=700,
    height=500,
)
fig.show()


## 5. Pipeline determinístico: interpretar um ACR

Um **ACR** aqui é um relato/caso clínico curto que o estudante pode editar. Montamos duas classes em sequência: `LLMClient` faz a chamada HTTP à API NVIDIA NIM (formato OpenAI-compatible) e `ACRInterpreter` pede JSON estruturado (`category`, `summary`, `urgency`). Sem LangChain — só `urllib` + `NVIDIA_API_KEY`.


In [ ]:
import os
import json
import urllib.request


class LLMClient:
    """Chamada vanilla à API NVIDIA NIM (formato OpenAI-compatible)."""

    def __init__(self):
        self.api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
        self.model = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-8b-instruct")

    def ask(self, messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        if not self.api_key:
            raise SystemExit("Defina NVIDIA_API_KEY e rode de novo.")
        body = json.dumps(
            {
                "model": self.model,
                "messages": messages,
                "temperature": 0,
                "response_format": {"type": "json_object"},
            }
        ).encode()
        req = urllib.request.Request(
            "https://integrate.api.nvidia.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"]


class ACRInterpreter:
    """Uma classe chama a outra: interpreta o ACR via LLMClient."""

    SYSTEM = (
        "Você é um assistente clínico. Dado um ACR (relato de caso), "
        "responda APENAS com JSON contendo as chaves: "
        "category (string), summary (string curta), urgency (baixa|media|alta)."
    )

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def interpret(self, acr: str) -> dict:
        raw = self.llm.ask(
            [
                {"role": "system", "content": self.SYSTEM},
                {"role": "user", "content": acr},
            ]
        )
        return json.loads(raw)


# Edite o ACR abaixo e rode a célula
acr = (
    "Paciente de 67 anos, hipertenso, chega com dor torácica em aperto "
    "há 40 minutos, sudorese e dispneia. PA 90/60 mmHg."
)

result = ACRInterpreter(LLMClient()).interpret(acr)
print(json.dumps(result, ensure_ascii=False, indent=2))


## 6. Docling + OCR (demonstração)

PDFs escaneados não têm camada de texto confiável — aí entra **OCR**. O Docling encapsula o pipeline: `PdfPipelineOptions(do_ocr=True)` + `DocumentConverter`. A célula abaixo é **só demonstração**: deixe `RUN_DOCLING = False` a menos que tenha um PDF/imagem local e as dependências de OCR instaladas.


In [ ]:
# Demonstração — NÃO rode em massa sem um arquivo local.
# Defina RUN_DOCLING = True e um caminho válido para experimentar.

RUN_DOCLING = False
PDF_PATH = "seu-arquivo-escaneado.pdf"  # PDF ou imagem

if not RUN_DOCLING:
    print("Demo Docling desligada. Defina RUN_DOCLING=True e PDF_PATH para extrair texto.")
else:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pipeline_options = PdfPipelineOptions(do_ocr=True)
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
        }
    )
    result = converter.convert(PDF_PATH)
    text = result.document.export_to_markdown()
    print(text[:2000])


## 7. LM Studio local (Nemotron 3 Nano 4B) — só demonstração

Para esta demo você precisa ter o **LM Studio** rodando localmente com o modelo **Nemotron 3 Nano 4B** carregado.
O servidor local (API compatível com o formato OpenAI — LM Studio) fica em `http://localhost:1234/v1`.

Esta célula **não faz parte da aula padrão** — é só demonstração. Mantenha `RUN_LM_STUDIO = False` a menos que o LM Studio esteja ativo com o modelo carregado.
Confira o **model id** exato no LM Studio (placeholder abaixo: `nvidia/Nemotron-Nano-4B`) e ajuste se necessário.


In [ ]:
# Demonstração — NÃO faz parte da aula padrão.
# Defina RUN_LM_STUDIO = True com o LM Studio rodando e Nemotron 3 Nano 4B carregado.

RUN_LM_STUDIO = False
LM_STUDIO_BASE = "http://localhost:1234/v1"
MODEL_ID = "nvidia/Nemotron-Nano-4B"  # ajuste ao id exato no LM Studio

if not RUN_LM_STUDIO:
    print("Demo LM Studio desligada. Defina RUN_LM_STUDIO=True com o servidor local ativo.")
else:
    import json
    import urllib.request

    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": "Diga olá em uma frase curta."},
        ],
        "temperature": 0.2,
        "max_tokens": 64,
    }
    req = urllib.request.Request(
        f"{LM_STUDIO_BASE}/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data["choices"][0]["message"]["content"])
